# Profile Original FINd Algorithm

Identify bottlenecks through cProfile and line_profiler

In [ ]:
# Load libraries
from find.FINd import FINDHasher
from find.FINd_opt import FINDHasher as FINDHasher_opt
from PIL import Image
import imagehash
import timeit
import time
import glob
import numpy as np
import matplotlib.pyplot as plt
import psutil
import os
import multiprocessing as mp

# Load FINd and test image
findHasher = FINDHasher()
img = Image.open("../data/test_image_1.jpg")

In [ ]:
# Compute pixel counts for all images in the dataset
all_paths = sorted(glob.glob("/data/meme_images/*.jpg"))
pixel_counts = []
for p in all_paths:
    with Image.open(p) as img_tmp:
        pixel_counts.append(np.prod(img_tmp.size))
pixel_counts = np.array(pixel_counts)

print(f"N images: {len(pixel_counts):,}")
print(f"\nPixel count (w × h):")
print(f"  Min:  {pixel_counts.min():,}")
print(f"  Max:  {pixel_counts.max():,}")
print(f"  Mean: {pixel_counts.mean():,.0f}")
print(f"  SD:   {pixel_counts.std():,.0f}")

## Core functions for benchmarking

In [ ]:
# Single image benchmarks
def cpu_per_call(fn, img, N=20, repeat=7):
    """Measure mean and SD of CPU time per call in milliseconds."""
    times = timeit.repeat(lambda: fn(img), number=N, repeat=repeat, timer=time.process_time)
    per_call = np.array(times) / N * 1000
    return per_call.mean(), per_call.std()

def _worker_memory(fn, img, queue):
    """Measure memory usage before and after hashing, and send 
    (delta, peak) in MiB to queue."""
    process = psutil.Process(os.getpid())

    before = process.memory_info().rss

    fn(img)

    after = process.memory_info().rss

    delta = (after - before) / (1024 ** 2)
    peak = after / (1024 ** 2)

    queue.put((delta, peak))

def measure_memory_mp(fn, img):
    """Generate a fresh subprocess to run _worker_memory, 
    ensuring a clean baseline for every call."""
    queue = mp.Queue()
    p = mp.Process(target=_worker_memory, args=(fn, img, queue))
    p.start()
    p.join()
    return queue.get()  # (delta, peak)

# Full dataset benchmarks
def _dataset_worker(fn, paths, sample_every, queue):
    """Hash every image in the dataset and record runtime, CPU time,
    and memory usage. Results are sent back via queue."""
    process = psutil.Process(os.getpid())
    memory_trace = []

    wall_start = time.time()
    cpu_start = time.process_time()

    for i, p in enumerate(paths):
        img = Image.open(p)
        fn(img)
        img.close()  # release file handle and pixel data immediately
        if i % sample_every == 0:  # sample RSS every 10 images to limit overhead
            memory_trace.append(process.memory_info().rss / 1024 ** 2)

    wall_elapsed = time.time() - wall_start
    cpu_elapsed = time.process_time() - cpu_start

    queue.put({
        'wall': wall_elapsed,
        'cpu': cpu_elapsed,
        'wall_per_image_ms': wall_elapsed / len(paths) * 1000,
        'cpu_per_image_ms': cpu_elapsed / len(paths) * 1000,
        'peak_memory': max(memory_trace),
        'memory_trace': memory_trace,
    })

def benchmark_dataset(fn, paths, sample_every=10):
    """Generate a fresh subprocess to run _dataset_worker and return its results."""
    queue = mp.Queue()
    p = mp.Process(target=_dataset_worker, args=(fn, paths, sample_every, queue))
    p.start()
    p.join()
    return queue.get()

## Profile using cProfile

Identify which parts of the function are slow on the level of the function:

In [ ]:
# Get the number of calls to each function and the time taken by each function
# sort the output by total time taken
%prun -s tottime findHasher.fromImage(img)

## Profile using line_profiler

Tells us which line(s) in the function are slow.

In [ ]:
%load_ext line_profiler
%lprun -f findHasher.fromImage -f findHasher.findHash256FromFloatLuma -f findHasher.fillFloatLumaFromBufferImage -f findHasher.boxFilter findHasher.fromImage(img)

# Computational Comparison

Compare the computational performance of FINd (original), FINd (optimized), and pHash algorithms on runtime, CPU time, memory, and scalability.

In [ ]:
# Load the optimized version of FINd
findHasher_opt = FINDHasher_opt()

# Load pHash
def phash(img):
    return imagehash.phash(img, hash_size=16)

# Define the algorithms to benchmark
algorithms_bench = {
    "FINd (original)":  findHasher.fromImage,
    "FINd (optimized)": findHasher_opt.fromImage,
    "pHash":            phash,
}

## Runtime and CPU time

In [ ]:
# Compute runtime and CPU time for each algorithm on the test image
# Print mean and SD of runtime and CPU time per call in milliseconds
print("FINd (original):")
r = %timeit -q -o findHasher.fromImage(img)
print(f"  Runtime: {r.average*1000:.2f} ± {r.stdev*1000:.2f} ms")
mean, std = cpu_per_call(findHasher.fromImage, img)
print(f"  CPU:  {mean:.2f} ± {std:.2f} ms")

print("\nFINd (optimized):")
r = %timeit -q -o findHasher_opt.fromImage(img)
print(f"  Runtime: {r.average*1000:.2f} ± {r.stdev*1000:.2f} ms")
mean, std = cpu_per_call(findHasher_opt.fromImage, img)
print(f"  CPU:  {mean:.2f} ± {std:.2f} ms")

print("\npHash:")
r = %timeit -q -o phash(img)
print(f"  Runtime: {r.average*1000:.2f} ± {r.stdev*1000:.2f} ms")
mean, std = cpu_per_call(phash, img)
print(f"  CPU:  {mean:.2f} ± {std:.2f} ms")

## Memory

In [ ]:
for name, fn in algorithms_bench.items():
    print(f"\nRunning {name}...")

    fn(img) # warm-up to avoid measuring one-time setup costs

    deltas = []
    peaks = []

    # Call measure_memory_mp multiple times to get reliable estimates of memory usage
    for _ in range(3): 
        delta, peak = measure_memory_mp(fn, img)
        deltas.append(delta)
        peaks.append(peak)

    print(f"{name}:")
    print(f"  Δ memory: {np.mean(deltas):.2f} ± {np.std(deltas):.2f} MiB")
    print(f"  Peak memory: {np.mean(peaks):.2f} ± {np.std(peaks):.2f} MiB")

## Scalability: runtime vs image size

In [ ]:
sizes = [128, 256, 512, 1024, 2048, 4096] # Image side length in pixels
N = 10 # Number of calls to each function for timing

find_orig_times = []
find_opt_times = []
phash_times = []

# Resize test image and compute runtime per algorithm
for size in sizes:
    resized = img.resize((size, size))

    t_orig = timeit.timeit(lambda: findHasher.fromImage(resized), number=N) / N * 1000
    t_opt = timeit.timeit(lambda: findHasher_opt.fromImage(resized), number=N) / N * 1000
    t_phash = timeit.timeit(lambda: phash(resized), number=N) / N * 1000

    find_orig_times.append(t_orig)
    find_opt_times.append(t_opt)
    phash_times.append(t_phash)
    print(f"{size}x{size}: FINd (original)={t_orig:.2f}ms  FINd (optimized)={t_opt:.2f}ms  pHash={t_phash:.2f}ms")

    resized.close()

# Plot figure with log(runtime per hash) vs image size across all three algorithms
plt.figure(figsize=(7, 4))
plt.plot(sizes, find_orig_times, marker="o", label="FINd (original)")
plt.plot(sizes, find_opt_times, marker="o", label="FINd (optimized)")
plt.plot(sizes, phash_times, marker="o", label="pHash")
plt.yscale("log")
plt.xlabel("Image side length (px)")
plt.ylabel("Runtime per hash (ms, log scale)")
plt.legend()
plt.tight_layout()
plt.savefig("scalability.png", dpi=300, bbox_inches="tight")
plt.show()

# Dataset-Level Performance

Run the optimized FINd and pHash algorithms across a sample of the full dataset and compare runtime, CPU time, and memory usage over images processed.

In [ ]:
# Select only the optimized FINd and pHash for benchmarking on the full dataset
algorithms_dataset = {k: v for k, v in algorithms_bench.items() if k in ["FINd (optimized)", "pHash"]}

# Benchmark FINd optimized and pHash on the full dataset and print results
dataset_results = {}
for name, fn in algorithms_dataset.items():
    print(f'Running {name}...')
    res = benchmark_dataset(fn, all_paths)
    dataset_results[name] = res
    print(f'  Runtime:  {res["wall"]:.2f}s  ({res["wall_per_image_ms"]:.2f} ms/image)')
    print(f'  CPU:   {res["cpu"]:.2f}s  ({res["cpu_per_image_ms"]:.2f} ms/image)')
    print(f'  Peak:  {res["peak_memory"]:.2f} MiB')